In [3]:
"""
ODIN v11.0: PRODUCTION-HARDENED SOVEREIGN ERP ORCHESTRATOR
Features: 
- Fraud Detection Node: Duplicate Invoices & Pricing Outliers (Fixed Hashing)
- Data Reliability Index (DRI) Governance
- Phantom Revenue & Shadow Payroll Detectors
- Resilient LangGraph DAG with Human-in-the-Loop Interrupts
- Enterprise Audit Logging & Multi-Page Verdict Reporting
"""

import os
import io
import json
import time
import xmlrpc.client
import pandas as pd
import matplotlib.pyplot as plt
from typing import Dict, Any, List, TypedDict, Optional
from datetime import datetime, timedelta

# LangChain & LangGraph
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, END

# PDF Generation (ReportLab)
from reportlab.lib.pagesizes import letter
from reportlab.platypus import (
    SimpleDocTemplate, Paragraph, Spacer, Table, 
    TableStyle, PageBreak, Image as RLImage
)
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib import colors
from reportlab.lib.units import inch
from reportlab.lib.enums import TA_CENTER

# ================================================================
# 1. STATE & CONFIGURATION
# ================================================================

class AgentState(TypedDict):
    """Sovereign State Schema for the Audit Workflow."""
    start_date: str
    end_date: str
    audit_data: Dict[str, Any]
    fraud_data: Dict[str, Any] # Fraud signals
    finance_data: Dict[str, Any]
    recovery_data: Dict[str, Any]
    reliability_index: Dict[str, float]
    is_approved: bool
    audit_trail: List[str]
    status: str

class ProductionConfig:
    USE_SIMULATION = True # Toggle for Live Odoo
    LLM_MODEL = "gpt-4-turbo"
    REPORT_DIR = "./odin_production/reports"
    LOG_DIR = "./odin_production/logs"
    RETRY_ATTEMPTS = 3
    
    # Fraud Thresholds
    DUPLICATE_TIME_WINDOW_DAYS = 2
    PRICING_VARIANCE_THRESHOLD = 0.15 # 15% deviation flags an outlier
    PAYROLL_ANOMALY_THRESHOLD = 10000.0 # Threshold for individual unlisted partner payments

for d in [ProductionConfig.REPORT_DIR, ProductionConfig.LOG_DIR]:
    os.makedirs(d, exist_ok=True)

# ================================================================
# 2. RESILIENT ERP CONNECTOR
# ================================================================

class OdooClient:
    """Production-grade client with retry logic and failover simulation."""
    def __init__(self):
        self.url = os.getenv('ODOO_URL', 'https://vendyltd.odoo.com/')
        self.db = os.getenv('ODOO_DB', 'vendyltd')
        self.user = os.getenv('ODOO_USER', 'muktadir@vendy.ltd')
        self.key = os.getenv('ODOO_PASSWORD', '') 
        self.connected = False
        if not ProductionConfig.USE_SIMULATION:
            self._connect()

    def _connect(self):
        for i in range(ProductionConfig.RETRY_ATTEMPTS):
            try:
                common = xmlrpc.client.ServerProxy(f"{self.url}/xmlrpc/2/common")
                self.uid = common.authenticate(self.db, self.user, self.key, {})
                self.models = xmlrpc.client.ServerProxy(f"{self.url}/xmlrpc/2/object")
                if self.uid:
                    self.connected = True; return
            except: time.sleep(2**i)

    def read(self, model, domain, fields):
        if ProductionConfig.USE_SIMULATION: return self._mock(model)
        if not self.connected: return []
        try:
            return self.models.execute_kw(self.db, self.uid, self.key, model, 'search_read', [domain], {'fields': fields})
        except: return []

    def _mock(self, model):
        """Simulation data designed to trigger all forensic nodes."""
        if model == 'account.move':
            return [
                # Fraud Signal: Exact Duplicates (Partner 1, Amount 5000, Same Date)
                {'id': 101, 'amount_total': 5000.0, 'date': '2025-12-15', 'partner_id': [1, 'Vendy Ltd'], 'move_type': 'out_invoice', 'payment_state': 'not_paid', 'amount_residual': 5000.0},
                {'id': 102, 'amount_total': 5000.0, 'date': '2025-12-15', 'partner_id': [1, 'Vendy Ltd'], 'move_type': 'out_invoice', 'payment_state': 'not_paid', 'amount_residual': 5000.0},
                # The $139k Major Debtor
                {'id': 103, 'amount_total': 139706.88, 'date': '2025-12-10', 'partner_id': [1, 'Vendy Ltd'], 'move_type': 'out_invoice', 'payment_state': 'not_paid', 'amount_residual': 139706.88},
                # Shadow Payroll Signal: Large payment to unlisted/generic partner
                {'id': 104, 'amount_total': 12500.0, 'date': '2025-12-12', 'partner_id': [99, 'Miscellaneous Services'], 'move_type': 'in_invoice', 'payment_state': 'paid', 'amount_residual': 0.0},
            ]
        if model == 'account.move.line':
            return [
                # Pricing Outlier Signal: Product 8 sold at 100 vs 45
                {'product_id': [8, 'FF Controller'], 'price_unit': 100.0, 'move_id': 101},
                {'product_id': [8, 'FF Controller'], 'price_unit': 45.0, 'move_id': 103},
            ]
        if model == 'stock.quant':
            return [{'quantity': -127930.0, 'product_id': [8, 'Ghost Stock']}]
        if model == 'sale.order':
            # Phantom Revenue Signal: Invoiced amount is significantly higher than Sales Orders
            return [{'amount_total': 150000.0}]
        return []

# ================================================================
# 3. CORE DAG NODES
# ================================================================

def audit_integrity_node(state: AgentState) -> Dict:
    """Primary Scan: Identifies Ghost Assets & Phantom Revenue."""
    client = OdooClient()
    sales = client.read('sale.order', [], ['amount_total'])
    invs = client.read('account.move', [('move_type', '=', 'out_invoice')], ['amount_total'])
    stock = client.read('stock.quant', [], ['quantity', 'product_id'])
    
    total_sales = sum(x['amount_total'] for x in sales)
    total_invoiced = sum(x['amount_total'] for x in invs)
    ghosts = [x for x in stock if x['quantity'] < 0]
    
    # Revenue Integrity Logic
    rev_gap = total_invoiced - total_sales
    dri = 1.0 - min(0.3, len(ghosts) * 0.1)
    
    # Penalize Phantom Revenue (Invoiced > Sold)
    if rev_gap > (total_sales * 0.1): # More than 10% gap
        dri -= 0.2

    return {
        "audit_data": {
            "count": len(ghosts), 
            "rev_gap": rev_gap,
            "total_sales": total_sales,
            "total_invoiced": total_invoiced
        },
        "reliability_index": {"system": round(max(0.1, dri), 2)},
        "audit_trail": state['audit_trail'] + [f"[{datetime.now()}] Base Integrity Scan Complete. Revenue Gap: ${rev_gap:,.2f}"]
    }

def fraud_detection_node(state: AgentState) -> Dict:
    """Forensic Fraud Scan: Duplicate Invoices & Pricing Inconsistencies."""
    client = OdooClient()
    invoices = client.read('account.move', [('move_type', '=', 'out_invoice')], ['amount_total', 'partner_id', 'date'])
    lines = client.read('account.move.line', [], ['product_id', 'price_unit', 'move_id'])
    
    fraud_flags = []
    
    # Process Invoices for Duplicates
    df_inv = pd.DataFrame(invoices)
    if not df_inv.empty:
        # Hashing Fix: Convert list to string
        df_inv['partner_id_str'] = df_inv['partner_id'].apply(lambda x: str(x) if isinstance(x, list) else x)
        dupes = df_inv[df_inv.duplicated(subset=['amount_total', 'partner_id_str', 'date'], keep=False)]
        if not dupes.empty:
            fraud_flags.append(f"CRITICAL: {len(dupes)} duplicate invoices found (Same Amount/Partner/Date).")

    # Process Lines for Pricing Outliers
    df_lines = pd.DataFrame(lines)
    pricing_issues = []
    if not df_lines.empty:
        df_lines['product_name'] = df_lines['product_id'].apply(lambda x: x[1] if isinstance(x, list) else str(x))
        for product_name, group in df_lines.groupby('product_name'):
            median_price = group['price_unit'].median()
            if median_price > 0:
                outliers = group[abs(group['price_unit'] - median_price) / median_price > ProductionConfig.PRICING_VARIANCE_THRESHOLD]
                if not outliers.empty:
                    pricing_issues.append(f"Outlier: Product '{product_name}' price variance exceeds 15%.")

    # Impact DRI
    current_dri = state['reliability_index']['system']
    fraud_penalty = (0.2 if fraud_flags else 0) + (0.1 if pricing_issues else 0)
    
    return {
        "fraud_data": {"flags": fraud_flags, "pricing": pricing_issues},
        "reliability_index": {"system": round(max(0.1, current_dri - fraud_penalty), 2)},
        "audit_trail": state['audit_trail'] + [f"[{datetime.now()}] Fraud Scan: Detected {len(fraud_flags)} duplicates and {len(pricing_issues)} price outliers."]
    }

def financial_forensic_node(state: AgentState) -> Dict:
    """Forensic Financial Analysis: Liquidity, QoE, and Shadow Payroll."""
    client = OdooClient()
    moves = client.read('account.move', [('state', '=', 'posted')], ['amount_total', 'move_type', 'payment_state', 'partner_id', 'amount_residual'])
    
    debtors = {}
    shadow_payroll = []
    rev, cash = 0, 0
    
    for m in moves:
        if m['move_type'] == 'out_invoice':
            rev += m['amount_total']
            if m['payment_state'] == 'paid': 
                cash += m['amount_total']
            elif m['partner_id']: 
                debtors[m['partner_id'][1]] = debtors.get(m['partner_id'][1], 0) + m['amount_residual']
        
        elif m['move_type'] == 'in_invoice':
            # Shadow Payroll Detector: Large payments to generic partners
            if m['amount_total'] > ProductionConfig.PAYROLL_ANOMALY_THRESHOLD:
                partner_name = m['partner_id'][1] if m['partner_id'] else "Unknown"
                if "Misc" in partner_name or "Service" in partner_name:
                    shadow_payroll.append(f"Shadow Payroll Alert: ${m['amount_total']:,.2f} to {partner_name}")

    qoe = cash / rev if rev > 0 else 1.0
    dri = state['reliability_index']['system']
    
    # Final Penalties
    if rev > 10000 and cash == 0: dri -= 0.4 # Paradox
    if shadow_payroll: dri -= 0.1 # Compliance risk

    return {
        "finance_data": {"profit": rev, "cash": cash, "debtors": debtors, "qoe": qoe, "shadow": shadow_payroll},
        "reliability_index": {"final": round(max(0.1, dri), 2)},
        "audit_trail": state['audit_trail'] + [f"[{datetime.now()}] Financial Forensic complete. Shadow Payroll checks: {len(shadow_payroll)} alerts."]
    }

def strategic_recovery_node(state: AgentState) -> Dict:
    """Drafts Actions and Safety Gateway."""
    llm = ChatOpenAI(model=ProductionConfig.LLM_MODEL, temperature=0)
    debtors = state['finance_data']['debtors']
    target = max(debtors, key=debtors.get) if debtors else "N/A"
    
    legal = "No action required."
    if debtors.get(target, 0) > 5000:
        legal = llm.invoke(f"Draft a formal legal demand for {target} regarding a ${debtors[target]:,.2f} debt. Use a final warning tone.").content

    return {
        "recovery_data": {"target": target, "debt": debtors.get(target, 0), "notice": legal},
        "status": "AWAITING_APPROVAL",
        "audit_trail": state['audit_trail'] + [f"[{datetime.now()}] Recovery plan drafted for top debtor: {target}"]
    }

def human_gate_node(state: AgentState) -> Dict:
    """The production safety gate."""
    print("\n" + "="*55 + "\nODIN SOVEREIGN GATEWAY: FORENSIC VERDICT\n" + "="*55)
    print(f"DATA RELIABILITY (DRI): {state['reliability_index']['final']*100}%")
    print(f"FRAUD ALERTS: {len(state['fraud_data']['flags'])}")
    print(f"SHADOW PAYROLL: {len(state['finance_data']['shadow'])}")
    
    choice = input("\nAUTHORIZE RECOVERY ACTIONS? (y/n): ").lower().strip()
    return {"is_approved": (choice == 'y'), "audit_trail": state['audit_trail'] + [f"[{datetime.now()}] Manual Authorization Decision: {'Approved' if choice == 'y' else 'Aborted'}"]}

# ================================================================
# 4. ORCHESTRATION & REPORTING
# ================================================================

def build_odin_v11():
    graph = StateGraph(AgentState)
    graph.add_node("audit", audit_integrity_node)
    graph.add_node("fraud", fraud_detection_node)
    graph.add_node("finance", financial_forensic_node)
    graph.add_node("plan", strategic_recovery_node)
    graph.add_node("gate", human_gate_node)
    
    graph.set_entry_point("audit")
    graph.add_edge("audit", "fraud")
    graph.add_edge("fraud", "finance")
    graph.add_edge("finance", "plan")
    graph.add_edge("plan", "gate")
    graph.add_edge("gate", END)
    return graph.compile()

class OdinReportEngine:
    @staticmethod
    def generate_pdf(state: AgentState):
        fpath = f"{ProductionConfig.REPORT_DIR}/ODIN_V11_FORENSIC_VERDICT.pdf"
        doc = SimpleDocTemplate(fpath, pagesize=letter)
        styles = getSampleStyleSheet()
        story = []

        # 1. Executive Branding
        title_style = ParagraphStyle('Title', parent=styles['Heading1'], fontSize=24, textColor=colors.navy, alignment=TA_CENTER)
        story.append(Paragraph("ODIN v11.0: SOVEREIGN FORENSIC VERDICT", title_style))
        story.append(Spacer(1, 20))

        # 2. Reliability & Fraud Dashboard
        dri = state['reliability_index']['final']
        color = colors.red if dri < 0.5 else colors.orange if dri < 0.8 else colors.green
        story.append(Paragraph(f"<b>DATA RELIABILITY (DRI):</b> <font color='{color}'>{dri*100}%</font>", styles['Heading2']))
        
        if state['fraud_data']['flags'] or state['finance_data']['shadow'] or state['fraud_data']['pricing']:
            story.append(Paragraph("<b>FORENSIC ALERTS:</b>", styles['Normal']))
            for flag in state['fraud_data']['flags']:
                story.append(Paragraph(f"• <font color='red'>{flag}</font>", styles['Normal']))
            for pricing in state['fraud_data']['pricing']:
                story.append(Paragraph(f"• <font color='orange'>{pricing}</font>", styles['Normal']))
            for shadow in state['finance_data']['shadow']:
                story.append(Paragraph(f"• <font color='red'>{shadow}</font>", styles['Normal']))
        
        # 3. Directives
        story.append(Spacer(1, 20))
        story.append(Paragraph("SOVEREIGN DIRECTIVES", styles['Heading2']))
        story.append(Paragraph(f"<b>TARGET:</b> {state['recovery_data']['target']} | <b>DEBT:</b> ${state['recovery_data']['debt']:,.2f}", styles['Normal']))
        story.append(Spacer(1, 10))
        
        legal_text = state['recovery_data']['notice'].replace('\n', '<br/>')
        story.append(Paragraph(legal_text, styles['Code']))

        doc.build(story)
        return fpath

if __name__ == "__main__":
    init_state = {
        "start_date": "2025-01-01", "end_date": "2025-12-31",
        "audit_data": {}, "fraud_data": {}, "finance_data": {}, "recovery_data": {},
        "reliability_index": {}, "is_approved": False, "status": "START",
        "audit_trail": [f"[{datetime.now()}] Forensic Engine Initialized."]
    }
    
    app = build_odin_v11()
    final_state = app.invoke(init_state)
    report = OdinReportEngine.generate_pdf(final_state)
    print(f"\n✅ SOVEREIGN FORENSIC VERDICT RENDERED: {report}")


ODIN SOVEREIGN GATEWAY: FORENSIC VERDICT
DATA RELIABILITY (DRI): 10.0%
FRAUD ALERTS: 1
SHADOW PAYROLL: 1

AUTHORIZE RECOVERY ACTIONS? (y/n): y

✅ SOVEREIGN FORENSIC VERDICT RENDERED: ./odin_production/reports/ODIN_V11_FORENSIC_VERDICT.pdf
